In [ ]:
import math
import random
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers # type: ignore
import numpy as np
import pandas_datareader as pdr
import tqdm
from tqdm import trange, tqdm
from collections import deque
import collections
import gym
import statistics
from matplotlib import pyplot as plt
from typing import Any, List, Sequence, Tuple

In [ ]:
# Valor epsilon pequeno para estabilizar operações de divisão
eps = np.finfo(np.float32).eps.item()

In [ ]:
env = gym.make("CartPole-v0")
seed = 42
env.seed(seed)
tf.random.set_seed(seed)
np.random.seed(seed)

/usr/local/lib/python3.10/dist-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(
/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/core.py:256: DeprecationWarning: WARN: Function `env.seed(seed)` is marked as deprecated and will be removed in the future. Please use `env.reset(seed=seed)` i

In [ ]:
def dataset_loader(stock_name):
  dataset = pdr.DataReader(
    stock_name, data_source="stooq")

  start_date = str(dataset.index[0]).split()[0]
  stop_date = str(dataset.index[-1]).split()[0]
  close = dataset['Close']

  return close

In [ ]:
stock_name = "AAPL"
data =  dataset_loader(stock_name)

In [ ]:
# data = tf.convert_to_tensor(data)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
if isinstance(data, tf.Tensor):
    try:
        data = tf.nest.map_structure(lambda x: x.numpy(), data)
    except AttributeError:
        # Erro ao converter o tensor
        raise TypeError("Tensor não pode ser convertido para um tipo que aceite .reset()")


In [ ]:
class IA_Trader(tf.keras.Model):
    def __init__(self, num_hidden_units:int, state_size, num_actions:int):

      super().__init__()

      self.state_size = state_size

      self.common = layers.Dense(num_hidden_units, activation="relu")
      self.actor = layers.Dense(num_actions)
      self.critic = layers.Dense(1)

    def call(self, inputs: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
      x = self.common(inputs)
      return self.actor(x), self.critic(x)

In [ ]:
num_hidden_units = 128
window_size = 10
data_sample = len(data) -1
num_actions = 3

model = IA_Trader(num_hidden_units, window_size, num_actions)

In [ ]:
def env_step(action: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
  """Retorna estado, recompensa e sinalizador concluído dada uma ação."""

  state, reward, done, _ = data.iloc[action]
  return (state.astype(np.float32),
          np.array(reward, np.int32),
          np.array(done, np.int32))

In [ ]:
def tf_env_step(action: tf.Tensor) -> List[tf.Tensor]:
  return tf.numpy_function(env_step, [action],
                           [tf.float32, tf.int32, tf.int32])

In [ ]:
def run_episode(
    initial_state: tf.Tensor,
    model: tf.keras.Model,
    max_steps: int) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
  """Executa um único episódio para coletar dados de treinamento."""

  action_probs = tf.TensorArray(dtype=tf.float32, size=0, dynamic_size=True)
  values = tf.TensorArray(dtype=tf.float32, size=0, dynamic_size=True)
  rewards = tf.TensorArray(dtype=tf.int32, size=0, dynamic_size=True)

  initial_state_shape = initial_state.shape
  state = initial_state

  for t in tf.range(max_steps):
    # Converter o estado em um tensor (batch size = 1)
    state = tf.expand_dims(state, 0)

    # Execute o modelo e obtenha probabilidades de ação e valor crítico
    action_logits_t, value = model(state)

    # Amostra da probabilidade da próxima ação
    action = tf.random.categorical(action_logits_t, 1)[0, 0]
    action_probs_t = tf.nn.softmax(action_logits_t)

    # Armazenar valores críticos
    values = values.write(t, tf.squeeze(value))

    # Armazenar log de probabilidade da ação escolhida
    action_probs = action_probs.write(t, action_probs_t[0, action])

    # Aplique ações ao meio ambiente para obter o próximo estado e recompensa
    state, reward, done = tf_env_step(action)
    state.set_shape(initial_state_shape)

    # Armazena recompensa
    rewards = rewards.write(t, reward)

    if tf.cast(done, tf.bool):
      break

  action_probs = action_probs.stack()
  values = values.stack()
  rewards = rewards.stack()

  return action_probs, values, rewards

In [ ]:
def get_expected_return(
    rewards: tf.Tensor,
    gamma: float,
    standardize: bool = True) -> tf.Tensor:
  """Calcule os retornos esperados por intervalo de tempo."""

  n = tf.shape(rewards)[0]
  returns = tf.TensorArray(dtype=tf.float32, size=n)

  # Start from the end of `rewards` and accumulate reward sums
  # into the `returns` array
  rewards = tf.cast(rewards[::-1], dtype=tf.float32)
  discounted_sum = tf.constant(0.0)
  discounted_sum_shape = discounted_sum.shape
  for i in tf.range(n):
    reward = rewards[i]
    discounted_sum = reward + gamma * discounted_sum
    discounted_sum.set_shape(discounted_sum_shape)
    returns = returns.write(i, discounted_sum)
  returns = returns.stack()[::-1]

  if standardize:
    returns = ((returns - tf.math.reduce_mean(returns)) /
               (tf.math.reduce_std(returns) + eps))

  return returns

In [ ]:
huber_loss = tf.keras.losses.Huber(reduction=tf.keras.losses.Reduction.SUM)

def compute_loss(
    action_probs: tf.Tensor,
    values: tf.Tensor,
    returns: tf.Tensor) -> tf.Tensor:
  """Computes the combined actor-critic loss."""

  advantage = returns - values

  action_log_probs = tf.math.log(action_probs)
  actor_loss = -tf.math.reduce_sum(action_log_probs * advantage)

  critic_loss = huber_loss(values, returns)

  return actor_loss + critic_loss

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)


@tf.function
def train_step(
    initial_state: tf.Tensor,
    model: tf.keras.Model,
    optimizer: tf.keras.optimizers.Optimizer,
    gamma: float,
    max_steps_per_episode: int) -> tf.Tensor:
  """Executa uma etapa de treinamento de modelo."""

  with tf.GradientTape() as tape:

    # Execute o modelo de um episódio para coletar dados de treinamento
    action_probs, values, rewards = run_episode(
        initial_state, model, max_steps_per_episode)

    # Calcule os retornos esperados
    returns = get_expected_return(rewards, gamma)

    # Converter dados de treinamento em formatos de tensor TF apropriados
    action_probs, values, returns = [
        tf.expand_dims(x, 1) for x in [action_probs, values, returns]]

    # Calculando valores de perdas para atualizar nossa rede
    loss = compute_loss(action_probs, values, returns)

  # Calcule os gradientes da perda
  grads = tape.gradient(loss, model.trainable_variables)

  # Aplique os gradientes aos parâmetros do modelo
  optimizer.apply_gradients(zip(grads, model.trainable_variables))

  episode_reward = tf.math.reduce_sum(rewards)

  return episode_reward